<a href="https://oscar-defelice.github.io">
<img src="../../img/image.png" height="125" align="right" />
</a>

# TP 02 — N-gram Language Models  *(SOLUTION)*

**Course:** Natural Language Processing  
**Session:** 2 / 8

> **Instructor only.** Do not distribute before the session.

---
## Part 0 — Imports & data loading

In [ ]:
from __future__ import annotations

import re
import string
import random
import math
from collections import defaultdict, Counter
from typing import Iterable, Sequence

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from datasets import load_dataset
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATASET_NAME = "ag_news"
LABEL_NAMES  = ["World", "Sports", "Business", "Sci/Tech"]

raw       = load_dataset(DATASET_NAME)
train_raw = raw["train"].to_pandas()
test_raw  = raw["test"].to_pandas()

train_df, val_df = train_test_split(
    train_raw, test_size=0.2, random_state=SEED, stratify=train_raw["label"]
)
test_df = test_raw.copy()
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

meta = {"label_names": LABEL_NAMES, "num_classes": len(LABEL_NAMES)}
print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

In [ ]:
def preprocess_text(text: str) -> str:
    """Normalise raw text: lowercase, strip HTML, remove punctuation, collapse spaces.

    Parameters
    ----------
    text : str
        Raw input string.

    Returns
    -------
    str
        Normalised string.
    """
    text = text.lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r" +", " ", text)
    return text.strip()


for df in (train_df, val_df, test_df):
    df["text_clean"] = df["text"].map(preprocess_text)

---
## Part 1 — Corpus statistics

In [ ]:
BOS = "<s>"
EOS = "</s>"
UNK = "<unk>"


def tokenize(text: str) -> list[str]:
    """Whitespace-split a preprocessed string into a list of tokens.

    Parameters
    ----------
    text : str
        Preprocessed (lowercase, no punctuation) text.

    Returns
    -------
    list[str]
        List of token strings.
    """
    return text.split()


# ── SOLUTION ──────────────────────────────────────────────────────────────────
train_tokens_list = [tokenize(t) for t in train_df["text_clean"]]
all_tokens        = [tok for sent in train_tokens_list for tok in sent]
unigram_counter   = Counter(all_tokens)

print(f"Total tokens (train) : {len(all_tokens):,}")
print(f"Vocabulary (all)     : {len(unigram_counter):,}")
print("Top-10 tokens:")
for tok, cnt in unigram_counter.most_common(10):
    print(f"  {tok:<20} {cnt:,}")

In [ ]:
def build_vocabulary(
    token_counter: Counter,
    min_freq: int = 2,
) -> set[str]:
    """Build a vocabulary by frequency thresholding.

    Tokens with count < min_freq are excluded from the vocabulary.
    Special tokens BOS, EOS, UNK are always included.

    Parameters
    ----------
    token_counter : Counter
        Token frequency counter built from the training corpus.
    min_freq : int
        Minimum count for a token to be included in the vocabulary.

    Returns
    -------
    set[str]
        Vocabulary set including special tokens.
    """
    vocab = {tok for tok, cnt in token_counter.items() if cnt >= min_freq}
    vocab.update({BOS, EOS, UNK})
    return vocab


# ── SOLUTION ──────────────────────────────────────────────────────────────────
vocab_full     = set(unigram_counter.keys()) | {BOS, EOS, UNK}
vocab_filtered = build_vocabulary(unigram_counter, min_freq=2)

print(f"Vocabulary (no threshold) : {len(vocab_full):,}")
print(f"Vocabulary (min_freq=2)   : {len(vocab_filtered):,}")
singletons = sum(1 for c in unigram_counter.values() if c == 1)
print(f"Hapax legomena removed    : {singletons:,}  ({100*singletons/len(vocab_full):.1f}% of types)")

---
## Part 2 — The `NgramLM` class

In [ ]:
class NgramLM:
    """N-gram language model with Laplace smoothing.

    Estimates $P(w_t | w_{t-n+1}, ..., w_{t-1})$ from a training corpus
    using maximum likelihood estimation with optional Laplace (add-alpha) smoothing.

    Attributes
    ----------
    n : int
        Order of the language model.
    alpha : float
        Laplace smoothing parameter.
    vocab : set[str]
        Vocabulary set. Set after calling ``fit``.
    counts : defaultdict[tuple[str, ...], Counter]
        N-gram counts keyed by (n-1)-gram context.
    context_totals : Counter
        Total count for each (n-1)-gram context.
    """

    def __init__(self, n: int = 2, alpha: float = 1.0) -> None:
        """Initialise the NgramLM.

        Parameters
        ----------
        n : int
            Order of the n-gram model.
        alpha : float
            Laplace smoothing pseudocount. Use 0.0 to disable smoothing.
        """
        if n < 1:
            raise ValueError(f"n must be >= 1, got {n}")
        self.n     = n
        self.alpha = alpha
        self.vocab: set[str] = set()
        self.counts: defaultdict = defaultdict(Counter)
        self.context_totals: Counter = Counter()

    # ------------------------------------------------------------------
    def _add_boundaries(self, tokens: list[str]) -> list[str]:
        """Wrap a token sequence with BOS/EOS boundary tokens.

        Parameters
        ----------
        tokens : list[str]
            Raw token sequence for one sentence.

        Returns
        -------
        list[str]
            Padded token sequence with boundary tokens.
        """
        return [BOS] * (self.n - 1) + tokens + [EOS]

    # ------------------------------------------------------------------
    def _unk_replace(self, tokens: list[str]) -> list[str]:
        """Replace out-of-vocabulary tokens with the UNK special token.

        Parameters
        ----------
        tokens : list[str]
            Token sequence.

        Returns
        -------
        list[str]
            Sequence with OOV tokens replaced by UNK.
        """
        return [t if t in self.vocab else UNK for t in tokens]

    # ------------------------------------------------------------------
    def fit(self, corpus: Iterable[list[str]], vocab: set[str]) -> "NgramLM":
        """Fit the language model on a tokenised corpus.

        Parameters
        ----------
        corpus : Iterable[list[str]]
            Iterable of tokenised sentences WITHOUT boundary tokens.
        vocab : set[str]
            Pre-built vocabulary.

        Returns
        -------
        NgramLM
            self.
        """
        self.vocab = vocab
        self.counts = defaultdict(Counter)
        self.context_totals = Counter()

        for sent in corpus:
            padded = self._add_boundaries(self._unk_replace(sent))
            for i in range(len(padded) - self.n + 1):
                context = tuple(padded[i : i + self.n - 1])
                token   = padded[i + self.n - 1]
                self.counts[context][token] += 1
                self.context_totals[context] += 1

        return self

    # ------------------------------------------------------------------
    def log_prob(
        self,
        token: str,
        context: tuple[str, ...],
    ) -> float:
        """Return the log-probability of a token given its context.

        Parameters
        ----------
        token : str
            The token whose probability is estimated.
        context : tuple[str, ...]
            The (n-1) preceding tokens.

        Returns
        -------
        float
            Natural log of the smoothed probability.

        Raises
        ------
        ValueError
            If context length does not match n-1.
        """
        if len(context) != self.n - 1:
            raise ValueError(
                f"Expected context length {self.n - 1}, got {len(context)}"
            )
        # Map OOV
        token   = token   if token   in self.vocab else UNK
        context = tuple(t if t in self.vocab else UNK for t in context)

        numerator   = self.counts[context][token] + self.alpha
        denominator = self.context_totals[context] + self.alpha * len(self.vocab)
        return math.log(numerator / denominator)

    # ------------------------------------------------------------------
    def sentence_log_prob(self, tokens: list[str]) -> float:
        """Compute the log-probability of a tokenised sentence.

        Parameters
        ----------
        tokens : list[str]
            Tokenised sentence WITHOUT boundary tokens.

        Returns
        -------
        float
            Total log-probability of the sentence.
        """
        padded = self._add_boundaries(self._unk_replace(tokens))
        total  = 0.0
        for i in range(len(padded) - self.n + 1):
            context = tuple(padded[i : i + self.n - 1])
            token   = padded[i + self.n - 1]
            total  += self.log_prob(token, context)
        return total

    # ------------------------------------------------------------------
    def perplexity(self, corpus: Iterable[list[str]]) -> float:
        """Compute perplexity of the model on a tokenised corpus.

        Parameters
        ----------
        corpus : Iterable[list[str]]
            Iterable of tokenised sentences WITHOUT boundary tokens.

        Returns
        -------
        float
            Perplexity. Lower is better.
        """
        total_log_prob = 0.0
        total_tokens   = 0
        for sent in corpus:
            total_log_prob += self.sentence_log_prob(sent)
            total_tokens   += len(sent) + 1  # +1 for EOS
        return math.exp(-total_log_prob / total_tokens)

In [ ]:
# ── Sanity checks ─────────────────────────────────────────────────────────────
_tiny_vocab = build_vocabulary(Counter(["the", "cat", "sat", "the", "cat", "mat", "on"]), min_freq=1)
_tiny_corpus = [
    ["the", "cat", "sat"],
    ["the", "cat", "sat", "on", "the", "mat"],
]

_lm = NgramLM(n=2, alpha=1.0).fit(_tiny_corpus, _tiny_vocab)

lp = _lm.log_prob("cat", ("the",))
assert lp <= 0
assert math.isfinite(_lm.log_prob("invisible_word", ("the",)))
ppl = _lm.perplexity(_tiny_corpus)
assert ppl > 1
assert _lm.sentence_log_prob(["the", "cat"]) < 0

print(f"All sanity checks passed. bigram PPL on tiny corpus: {ppl:.2f}")

---
## Part 3 — Smoothing

In [ ]:
# ── SOLUTION ──────────────────────────────────────────────────────────────────
vocab = build_vocabulary(unigram_counter, min_freq=2)

lm_bigram_raw    = NgramLM(n=2, alpha=0.0).fit(train_tokens_list, vocab)
lm_bigram_smooth = NgramLM(n=2, alpha=1.0).fit(train_tokens_list, vocab)

print(f"Bigram model (raw)    trained. Total contexts: {len(lm_bigram_raw.counts):,}")
print(f"Bigram model (smooth) trained. |V| = {len(vocab):,}")

In [ ]:
def demonstrate_zero_count(
    lm_no_smooth: NgramLM,
    lm_smooth: NgramLM,
    test_sentence: list[str],
) -> None:
    """Show the effect of a zero-count bigram on sentence log-probability.

    Parameters
    ----------
    lm_no_smooth : NgramLM
        Bigram LM trained without Laplace smoothing (alpha=0).
    lm_smooth : NgramLM
        Bigram LM trained with Laplace smoothing (alpha=1).
    test_sentence : list[str]
        Tokenised sentence that contains at least one unseen bigram.
    """
    padded = lm_smooth._add_boundaries(
        lm_smooth._unk_replace(test_sentence)
    )
    n_zero = 0
    print(f"{'Context':<25} {'Token':<20} {'log_p (raw)':>14} {'log_p (smooth)':>16}")
    print("-" * 78)
    for i in range(len(padded) - 1):
        ctx   = (padded[i],)
        token = padded[i + 1]
        raw_lp = lm_no_smooth.counts[ctx][token]  # raw count
        # For raw: if count == 0 and alpha == 0, log(0) = -inf
        if raw_lp == 0:
            lp_raw = float("-inf")
            n_zero += 1
            marker = " <-- zero count"
        else:
            lp_raw = lm_no_smooth.log_prob(token, ctx)
            marker = ""
        lp_sm = lm_smooth.log_prob(token, ctx)
        print(f"{str(ctx):<25} {token:<20} {lp_raw:>14.4f} {lp_sm:>16.4f}{marker}")
    print(f"\nUnseen bigrams: {n_zero} / {len(padded)-1}")


_test_sent = tokenize(val_df.iloc[0]["text_clean"])
demonstrate_zero_count(lm_bigram_raw, lm_bigram_smooth, _test_sent)

---
## Part 4 — Evaluation

In [ ]:
def train_ngram_lm(
    train_df: pd.DataFrame,
    text_col: str,
    n: int,
    alpha: float,
    min_freq: int = 2,
) -> NgramLM:
    """Convenience wrapper: build vocabulary and fit an NgramLM in one call.

    Parameters
    ----------
    train_df : pd.DataFrame
        Training dataframe with a preprocessed text column.
    text_col : str
        Column containing preprocessed text strings.
    n : int
        N-gram order.
    alpha : float
        Laplace smoothing parameter.
    min_freq : int
        Minimum token frequency for vocabulary inclusion.

    Returns
    -------
    NgramLM
        Fitted NgramLM instance.
    """
    corpus  = [tokenize(t) for t in train_df[text_col]]
    counter = Counter(tok for sent in corpus for tok in sent)
    vocab   = build_vocabulary(counter, min_freq=min_freq)
    return NgramLM(n=n, alpha=alpha).fit(corpus, vocab)


# ── SOLUTION ──────────────────────────────────────────────────────────────────
test_corpus  = [tokenize(t) for t in test_df["text_clean"]]

ppl_by_n: dict[int, float] = {}
lms: dict[int, NgramLM] = {}
for n in [1, 2, 3]:
    lm = train_ngram_lm(train_df, "text_clean", n=n, alpha=1.0)
    ppl = lm.perplexity(test_corpus)
    ppl_by_n[n] = ppl
    lms[n] = lm
    print(f"{n}-gram LM | PPL = {ppl:.1f}")

In [ ]:
def compute_per_class_ppl(
    lm: NgramLM,
    test_df: pd.DataFrame,
    text_col: str,
    label_col: str,
    label_names: list[str],
) -> pd.DataFrame:
    """Compute perplexity of an NgramLM separately for each class.

    Parameters
    ----------
    lm : NgramLM
        Fitted language model.
    test_df : pd.DataFrame
        Test dataframe.
    text_col : str
        Preprocessed text column.
    label_col : str
        Integer label column.
    label_names : list[str]
        Human-readable class names.

    Returns
    -------
    pd.DataFrame
        Columns: ["class", "n_docs", "perplexity"].
    """
    rows = []
    for class_id, name in enumerate(label_names):
        subset  = test_df[test_df[label_col] == class_id]
        corpus  = [tokenize(t) for t in subset[text_col]]
        ppl     = lm.perplexity(corpus)
        rows.append({"class": name, "n_docs": len(subset), "perplexity": round(ppl, 1)})
    return pd.DataFrame(rows)


# ── SOLUTION ──────────────────────────────────────────────────────────────────
per_class_ppl = compute_per_class_ppl(
    lms[2], test_df, "text_clean", "label", LABEL_NAMES
)
display(per_class_ppl.style.background_gradient(subset=["perplexity"], cmap="RdYlGn_r"))

In [ ]:
def plot_ppl_comparison(
    ppl_by_n: dict[int, float],
    title: str = "Perplexity vs n-gram order (AG News test set)",
) -> None:
    """Bar chart comparing perplexity across n-gram orders.

    Parameters
    ----------
    ppl_by_n : dict[int, float]
        Mapping from n-gram order to perplexity value.
    title : str
        Plot title.
    """
    ns   = sorted(ppl_by_n)
    ppls = [ppl_by_n[n] for n in ns]
    labels = [f"{n}-gram" for n in ns]

    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar(labels, ppls, color=["steelblue", "seagreen", "darkorange"], width=0.5)
    ax.bar_label(bars, fmt="%.0f", padding=4, fontsize=11)
    ax.set_ylabel("Perplexity (lower is better)")
    ax.set_title(title)
    ax.set_ylim(0, max(ppls) * 1.2)
    plt.tight_layout()
    plt.show()


plot_ppl_comparison(ppl_by_n)

---
## Part 5 — Text generation

In [ ]:
def generate(
    lm: NgramLM,
    max_tokens: int = 30,
    temperature: float = 1.0,
    seed: int | None = None,
) -> str:
    """Generate a sentence by sampling from the n-gram LM.

    Parameters
    ----------
    lm : NgramLM
        Fitted NgramLM.
    max_tokens : int
        Maximum tokens to generate.
    temperature : float
        Sampling temperature. Must be > 0.
    seed : int or None
        Random seed.

    Returns
    -------
    str
        Generated sentence as a space-joined string.
    """
    if temperature <= 0:
        raise ValueError(f"temperature must be > 0, got {temperature}")
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    context   = tuple([BOS] * (lm.n - 1))
    generated: list[str] = []

    for _ in range(max_tokens):
        # Candidate tokens: everything with a count in this context + full vocab for smoothing
        candidates = sorted(
            lm.vocab - {BOS}  # never generate BOS mid-sentence
        )
        log_probs = np.array([lm.log_prob(tok, context) for tok in candidates])
        log_probs /= temperature
        # Numerically stable softmax
        log_probs -= log_probs.max()
        probs      = np.exp(log_probs)
        probs     /= probs.sum()

        token = np.random.choice(candidates, p=probs)
        if token == EOS:
            break
        generated.append(token)
        context = tuple(list(context[1:]) + [token]) if lm.n > 1 else ()

    return " ".join(generated)


print("Bigram sample:")
print(generate(lms[2], seed=SEED))

In [ ]:
# ── SOLUTION: class-conditioned generation ───────────────────────────────────
print("Class-conditioned generation (bigram, temperature=1.0)\n")
for class_id, name in enumerate(LABEL_NAMES):
    class_df = train_df[train_df["label"] == class_id]
    lm_class = train_ngram_lm(class_df, "text_clean", n=2, alpha=1.0)
    print(f"[{name}]")
    for i in range(3):
        print(f"  {i+1}. {generate(lm_class, seed=SEED+i)}")
    print()

In [ ]:
def temperature_sweep(
    lm: NgramLM,
    temperatures: list[float],
    n_samples: int = 3,
    seed: int = SEED,
) -> None:
    """Print generated sentences at multiple temperature values.

    Parameters
    ----------
    lm : NgramLM
        Fitted NgramLM.
    temperatures : list[float]
        List of temperature values to sweep.
    n_samples : int
        Number of sentences to generate per temperature.
    seed : int
        Initial random seed.
    """
    for temp in temperatures:
        print(f"Temperature = {temp}")
        for i in range(n_samples):
            print(f"  {generate(lm, temperature=temp, seed=seed + i)}")
        print()


temperature_sweep(lms[2], temperatures=[0.5, 1.0, 2.0])

---
## Part 5.5 — Homework solution: Naive Bayes

In [ ]:
from scipy.sparse import spmatrix
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import classification_report, f1_score


class NaiveBayes:
    """Multinomial Naive Bayes classifier with Laplace smoothing.

    Attributes
    ----------
    alpha : float
        Laplace smoothing pseudocount.
    classes_ : np.ndarray
        Sorted array of unique class labels.
    log_priors_ : np.ndarray
        Log-prior probabilities, shape (n_classes,).
    log_likelihoods_ : np.ndarray
        Log-word likelihoods, shape (n_classes, n_features).
    """

    def __init__(self, alpha: float = 1.0) -> None:
        """Initialise NaiveBayes.

        Parameters
        ----------
        alpha : float
            Laplace smoothing pseudocount.
        """
        self.alpha = alpha
        self.classes_: np.ndarray | None = None
        self.log_priors_: np.ndarray | None = None
        self.log_likelihoods_: np.ndarray | None = None

    def fit(self, X: spmatrix, y: np.ndarray) -> "NaiveBayes":
        """Estimate model parameters from labelled training data.

        Parameters
        ----------
        X : sparse matrix, shape (n_samples, n_features)
            Bag-of-words count matrix.
        y : np.ndarray, shape (n_samples,)
            Integer class labels.

        Returns
        -------
        NaiveBayes
            self.
        """
        self.classes_ = np.unique(y)
        n_classes  = len(self.classes_)
        n_features = X.shape[1]

        # Log priors
        class_counts = np.array([(y == c).sum() for c in self.classes_], dtype=float)
        self.log_priors_ = np.log(class_counts / class_counts.sum())

        # Log likelihoods with Laplace smoothing
        self.log_likelihoods_ = np.zeros((n_classes, n_features))
        for i, c in enumerate(self.classes_):
            mask         = y == c
            word_counts  = np.asarray(X[mask].sum(axis=0)).flatten()
            smoothed     = word_counts + self.alpha
            self.log_likelihoods_[i] = np.log(smoothed / smoothed.sum())

        return self

    def predict_log_proba(self, X: spmatrix) -> np.ndarray:
        """Compute unnormalised log-posterior scores.

        Parameters
        ----------
        X : sparse matrix, shape (n_samples, n_features)
            Bag-of-words count matrix.

        Returns
        -------
        np.ndarray, shape (n_samples, n_classes)
            Unnormalised log-posterior matrix.
        """
        return X @ self.log_likelihoods_.T + self.log_priors_

    def predict(self, X: spmatrix) -> np.ndarray:
        """Predict the most probable class for each sample.

        Parameters
        ----------
        X : sparse matrix, shape (n_samples, n_features)
            Bag-of-words count matrix.

        Returns
        -------
        np.ndarray, shape (n_samples,)
            Predicted class labels.
        """
        return self.classes_[np.argmax(self.predict_log_proba(X), axis=1)]


# ── SOLUTION: train and evaluate ─────────────────────────────────────────────
vec = CountVectorizer(max_features=50_000)
X_train = vec.fit_transform(train_df["text_clean"])
X_val   = vec.transform(val_df["text_clean"])
X_test  = vec.transform(test_df["text_clean"])

y_train = train_df["label"].values
y_test  = test_df["label"].values

nb = NaiveBayes(alpha=1.0).fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

print("Naive Bayes — AG News test set")
print(classification_report(y_test, y_pred_nb, target_names=LABEL_NAMES))

macro_f1 = f1_score(y_test, y_pred_nb, average="macro")
print(f"Macro F1: {macro_f1:.4f}")

---
## 🗒 Instructor Notes

### Expected results on AG News

**Part 4 — Perplexity**

| Model | Test PPL (approx) |
|-------|-------------------|
| Unigram | 1,800 -- 2,500 |
| Bigram  | 350 -- 500     |
| Trigram | 150 -- 250     |

Exact values depend on the min_freq threshold and vocabulary handling. Unigram PPL is dominated by the
vocabulary size and Zipf distribution. Trigram PPL is lower but sparsity is severe -- Laplace smoothing
transfers significant probability mass to unseen trigrams.

**Per-class PPL (bigram):**
Sports typically has the lowest PPL (~300--400) because its vocabulary is more restricted
(team names, sport-specific verbs) and sentence structure is formulaic. Sci/Tech tends to have
the highest PPL due to technical jargon and product names creating many hapax legomena.

**Part 5 -- Generation:**
Bigram output is mostly incoherent but will show class-typical words.
Trigram output is slightly more fluent locally but still breaks at 4+ word spans.
At temperature < 0.5, models will output repetitive n-gram loops (expected -- use as teaching moment).

**Homework NB:**
Expected macro F1 on AG News test set: **0.89 -- 0.91**.
Per-class: Sports and Business are the easiest (F1 ~ 0.93--0.95), World and Sci/Tech the hardest
due to vocabulary overlap (confirmed by the Jaccard analysis in TP01).

### Common student mistakes

1. **Forgetting to add boundaries before counting.** Context `(BOS, w1)` should appear for the
   first word in every sentence. If boundaries are omitted, the first word has no valid context
   and `log_prob` will use the smoothing floor for every sentence start.
2. **Counting EOS as a predicted token but not adding it to the denominator count.**
   The total token count in `perplexity` must include the EOS token for comparability.
3. **UNK replacement before vs after boundary addition.**
   BOS and EOS must never be replaced with UNK. Apply `_unk_replace` before `_add_boundaries`.
4. **In `generate`: drawing from a fixed list of seen tokens only.**
   With small corpora or class-conditioned models, the seen token set may not contain EOS
   in every context, causing infinite loops. Sampling over the full vocab with smoothed
   probabilities guarantees termination.
5. **NaiveBayes: fitting the CountVectorizer on train+test.**
   Classic leakage. The vectoriser must be fit only on `train_df`.
6. **NaiveBayes: not working in log-space.**
   Students who compute `P(x|y) = prod P(w_i|y)` directly will get underflow for most
   documents and all predictions will collapse to the same class.